# **EDA Notebook**



---
## 0. Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
# <Student to fill this section>
group_name = ""
student_name = ""
student_id = ""

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section>

### 0.b Import Packages

In [ ]:
import pandas as pd
import altair as alt
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

---
## B. Data Understanding

In [ ]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "customer.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [ ]:
# Structural overview: shape, dtypes, head/tail, missing values, duplicates.
print(f'Dataset shape : {df.shape[0]:,} rows x {df.shape[1]} columns\n')
df.info()

print('\nFirst 5 rows:')
display(df.head())
print('\nLast 5 rows:')
display(df.tail())

In [ ]:
# Missing values, duplicates, and per-column uniqueness.
print('Missing values per column:')
miss = df.isna().sum().to_frame('n_missing')
miss['pct_missing'] = (df.isna().mean() * 100).round(2)
display(miss)

print(f'\nExact duplicate rows           : {df.duplicated().sum()}')
print(f'Unique customer_id values      : {df["customer_id"].nunique():,} / {len(df):,}')
print(f'Unique account_number values   : {df["account_number"].nunique():,} / {len(df):,}')

In [ ]:
dataset_insights = """
Dataset: `customer.csv` — 14,275 rows x 5 columns. This is the customer dimension table.

Columns (all stored as strings):
  - customer_id    : UUID. Intended as primary key (see B.2 for caveat).
  - person_id      : FK to person dimension; populated for INDIVIDUAL customers.
  - store_id       : FK to store dimension; populated for BUSINESS customers.
  - territory_id   : FK to sales territory dimension.
  - account_number : human-readable account code (e.g. 'AW00019475').

Quality / completeness:
  - DATA QUALITY ISSUE: 3,944 exact-duplicate rows. Only 10,331 unique
    customer_id values across 14,275 rows. This must be deduplicated
    before any aggregation (the prep notebook handles this in B.2 via
    drop_duplicates on customer_id).
  - DATA QUALITY ISSUE: account_number is NOT one-to-one with customer_id
    in the raw file — 939 account_numbers appear against multiple customer_id
    rows (see B.2). After dedup the relationship is one-to-one as expected.
  - customer_id, territory_id and account_number are otherwise fully populated.
  - person_id is missing for 237 rows (1.66%) and store_id for 13,823 rows
    (96.83%). These are NOT data errors — they are STRUCTURAL NULLs that
    encode the customer type (individual vs business). See B.3.

Implications for the repeat-purchase use case:
  - The table is small enough to load entirely into memory and merge.
  - Deduplication is non-negotiable. Without it, every customer-level
    aggregate (RFM, recent activity, etc.) inflates by the duplication
    multiplier for affected customers.
  - All IDs are UUIDs; they carry no business meaning and cannot be used
    as model features directly.
  - The dominant business segment by row-count is individuals (~97%),
    so global model performance will be largely driven by the individual
    sub-population unless explicitly stratified.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `customer_id`

In [ ]:
# customer_id is the table's primary key. Validate format and uniqueness.
uuid_re = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')

is_valid = df['customer_id'].astype(str).str.match(uuid_re)
lengths  = df['customer_id'].astype(str).str.len()

print(f'Total customer_id values   : {len(df):,}')
print(f'Unique customer_id values  : {df["customer_id"].nunique():,}')
print(f'Missing customer_id values : {df["customer_id"].isna().sum():,}')
print(f'Proportion matching UUID v-format: {is_valid.mean():.0%}')
print(f'String length stats:')
print(lengths.describe().round(2))

In [ ]:
# Confirm one-to-one alignment between customer_id and account_number.
# If they are perfectly aligned, account_number is redundant for joins.
n_pairs = df.groupby('customer_id')['account_number'].nunique()
print(f'Customers with >1 account_number : {(n_pairs > 1).sum()}')

rev_pairs = df.groupby('account_number')['customer_id'].nunique()
print(f'Account_numbers with >1 customer_id : {(rev_pairs > 1).sum()}')

In [ ]:
feature_1_insights = """
`customer_id` is the intended primary key of the customer table.

Distribution / format:
  - 14,275 total values, 10,331 unique (72.4%). 3,944 customer_id values
    appear in more than one row — these are the exact-duplicate rows
    surfaced in B.1.
  - Every value matches the canonical UUID v-format (8-4-4-4-12 hex with
    hyphens, length 36). No malformed IDs.
  - Per-customer: each customer_id has exactly one account_number (0
    violations). The reverse holds in the OPPOSITE direction for 939
    account_numbers, which means those account_numbers appear in multiple
    DUPLICATE customer rows. After deduplication on customer_id the
    relationship becomes one-to-one in both directions, and account_number
    is then informationally redundant with customer_id.

Limitations / role:
  - As a UUID, customer_id has no inherent ordering or meaning. It is
    suitable only as a join key, not as a model feature — a model that
    used the raw UUID would memorise customers rather than generalise.
  - Duplication must be resolved before joining onto the sales fact
    table, or each duplicated customer would contribute their order
    history multiple times. The prep notebook deduplicates on customer_id
    with keep='first' in section B.2.
  - Downstream prep code uses customer_id ONLY to join sales_order_header
    to customer attributes and to build the repeat-purchase label; it is
    dropped from the feature matrix before modelling.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `person_id` and `store_id` (customer type)

In [ ]:
# person_id is populated for individual customers; store_id for businesses.
# Cross-tabulate the NULL patterns to confirm the structural meaning of NULLs.
has_person = df['person_id'].notna()
has_store  = df['store_id'].notna()

type_mix = pd.crosstab(has_person.rename('has_person_id'),
                       has_store.rename('has_store_id'),
                       margins=True, margins_name='total')
display(type_mix)

individuals = (has_person & ~has_store).sum()
businesses  = (~has_person & has_store).sum()
both        = (has_person & has_store).sum()
neither     = (~has_person & ~has_store).sum()

print(f'\nIndividual customers (person only)   : {individuals:>5,} ({individuals/len(df):.2%})')
print(f'Business customers   (store only)    : {businesses:>5,} ({businesses/len(df):.2%})')
print(f'Both person AND store present        : {both:>5,} ({both/len(df):.2%})')
print(f'Neither person NOR store present     : {neither:>5,} ({neither/len(df):.2%})')

In [ ]:
# Visualise the customer-type split (the derived feature most relevant
# to the repeat-purchase model).
import pandas as pd
type_df = pd.DataFrame({
    'customer_type': ['individual (person only)', 'business (store only)',
                      'both present', 'neither present'],
    'count'        : [individuals, businesses, both, neither],
})

alt.Chart(type_df).mark_bar().encode(
    x=alt.X('count:Q', title='Number of customers'),
    y=alt.Y('customer_type:N', sort='-x', title=None),
    tooltip=['customer_type', 'count']
).properties(width=500, height=180,
             title='Customer type breakdown (derived from person_id / store_id NULLs)')

In [ ]:
feature_2_insights = """
`person_id` and `store_id` are STRUCTURALLY-NULL columns whose NULL/non-NULL
pattern encodes the customer type.

Joint distribution (before deduplication of customer_id):
  - Individual customers (person_id present, store_id NULL): 13,823 (96.83%).
  - Business   customers (store_id present,  person_id NULL):    237 (1.66%).
  - Customers with BOTH columns populated                  :    215 (1.51%).
  - Customers with NEITHER column populated                :      0.

Data-quality findings:
  - These NULLs are NOT missing data. They CARRY information ('this customer
    is an individual / a business / both') and must NOT be imputed away.
  - 215 customers have BOTH person_id AND store_id populated. This appears
    to be a legitimate (small) segment — likely sole-trader / owner-operator
    accounts where the same entity is both an individual person and a
    business store. The prep notebook treats `is_individual` and
    `is_business` as two SEPARATE binary flags rather than a single
    mutually-exclusive enum, so this 1.5% segment can be modelled
    explicitly.

Implications for repeat-purchase modelling:
  - The dataset is heavily skewed toward individual customers (~97%).
    A global model will predominantly learn individual behaviour.
  - The 237 store-only and 215 dual-flag customers, while only ~3.2% of
    rows, may contribute a much larger share of revenue (one B2B order
    can equal hundreds of consumer orders — confirmed in the
    sales_order_header EDA's bimodal money distribution). Whether they
    should be weighted differently in the loss function depends on the
    downstream business metric the model is optimised for.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `territory_id`

In [ ]:
# Customers are distributed across a small number of territories. Each
# territory is identified by a UUID, so we count and rank them by row share.
ter_counts = df['territory_id'].value_counts().reset_index()
ter_counts.columns = ['territory_id', 'n_customers']
ter_counts['short_id']   = ter_counts['territory_id'].str[:8]
ter_counts['pct_share']  = ter_counts['n_customers'] / len(df)

print(f'Number of distinct territories : {ter_counts.shape[0]}')
print(f'Top-1 territory share          : {ter_counts["pct_share"].iloc[0]:.2%}')
print(f'Top-3 territories combined     : {ter_counts["pct_share"].head(3).sum():.2%}')
display(ter_counts.head(10).style.format({'pct_share': '{:.2%}'}))

In [ ]:
# Bar chart of customer counts per territory, sorted descending.
alt.Chart(ter_counts).mark_bar().encode(
    x=alt.X('n_customers:Q', title='Number of customers'),
    y=alt.Y('short_id:N', sort='-x', title='Territory (first 8 chars of UUID)'),
    tooltip=['territory_id', 'n_customers',
             alt.Tooltip('pct_share:Q', format='.2%')]
).properties(width=480, height=260,
             title='Customer distribution across territories')

In [ ]:
feature_n_insights = """
`territory_id` is the geography dimension key for each customer.

Distribution:
  - Fully populated, no missing values.
  - Only 4 distinct territories — substantially lower cardinality than a
    typical AdventureWorks deployment (which carries 10 territories at
    sales-territory level). The retailer in this dataset is concentrated
    in a small number of regions.
  - Distribution is moderately concentrated: the top territory holds
    ~39% of customers and the top three combined hold ~80%. The 4th
    territory is a long-tail with ~20% of customers spread thinly.
  - Low cardinality makes territory_id a strong candidate for one-hot
    encoding (handled in the prep notebook's Section E.3). After
    one-hot encoding, territory contributes only 4 columns to the model.

Implications for repeat-purchase modelling:
  - Territory captures regional differences in customer behaviour
    (purchasing power, channel mix, seasonality). It is retained as a
    model feature, not dropped.
  - With only 4 territories the model's geographic-effect coefficients
    will be relatively stable (vs a long-tailed territory column where
    rare territories would be under-supported and noisy).
  - The sales_order_header table also carries a territory_id (the
    territory of the order, not necessarily the customer's home
    territory). Reconciling the two is handled in prep B.4: if an order's
    territory_id is missing it is back-filled from the customer dimension.

Caveat:
  - Encoding territory as a UUID hides any natural geographic adjacency.
    A model cannot tell that two neighbouring territories are 'similar'.
    If geographic clustering matters downstream, a richer geography feature
    (continent, country) would be needed — out of scope here.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)